# Sentinel-2 via Google Earth Engine

Downloading raw SAFE scenes one by one (Copernicus Browser/CDSE) is slow and heavy, so this uses Google Earth Engine instead: cloud masking and compositing happen server-side, and we only pull down the final composite.

**Requires**: a registered GEE account/project (already set up here, project `main-test-508113`) - see https://code.earthengine.google.com/ and https://console.cloud.google.com/earth-engine.

**What this notebook does**, per year from 2020-2024:
1. Filters `COPERNICUS/S2_SR_HARMONIZED` (Sentinel-2 L2A) to the Banten AOI, dry season (June-September), cloud cover < 20%
2. Cloud-masks using the SCL band (drops cloud shadow, medium/high probability cloud, cirrus)
3. Takes the median composite per year, bands B2/B3/B4/B8/B11/B12
4. Computes NDVI, EVI, NDMI
5. Exports the result (GeoTIFF, EPSG:32748 / UTM 48S, 10m) to **Google Drive** (folder `GeoSuitability_Sentinel2`) - the export runs asynchronously on Google's servers, the notebook doesn't block waiting for it

This only covers what the main model (CNN-MLP fusion) needs - one representative composite per year/growing season. Monthly composites would only be needed for the optional LSTM branch (see the project notes).

In [ ]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

## 1. Authenticate with Earth Engine

In [ ]:
import ee

# Update this if your GEE project ID is different (shown top-left on
# console.cloud.google.com)
GEE_PROJECT = "main-test-508113"

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="notebook")  # notebook mode: paste a token back manually, more reliable than the local-server callback
    ee.Initialize(project=GEE_PROJECT)

print("Earth Engine ready, project:", GEE_PROJECT)

## 2. Load the Banten AOI

In [ ]:
import geopandas as gpd

from common import RAW_DIR

aoi_path = RAW_DIR / "boundaries" / "aoi_banten.geojson"
gdf = gpd.read_file(aoi_path)

# light simplification (~100m) so the geometry isn't too heavy for GEE, still accurate enough for an AOI
geom = gdf.geometry.iloc[0].simplify(0.001)
aoi = ee.Geometry(geom.__geo_interface__)

print("Banten AOI loaded, area roughly:", aoi.area().divide(1e6).getInfo(), "km2")

## 3. Pipeline: filter, cloud-mask, composite, vegetation indices

In [ ]:
CLOUD_COVER_MAX = 20
YEAR_START = 2020
YEAR_END = 2024
DRY_SEASON = ("06-01", "09-30")  # dry season: June-September
EXPORT_CRS = "EPSG:32748"  # UTM 48S, matches the project's grid convention
EXPORT_SCALE = 10  # meters
DRIVE_FOLDER = "GeoSuitability_Sentinel2"

S2_BANDS = ["B2", "B3", "B4", "B8", "B11", "B12"]


def mask_s2_clouds(image):
    scl = image.select("SCL")
    # drop: 3=cloud shadow, 8=cloud medium prob, 9=cloud high prob, 10=cirrus
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    optical = image.select("B.*").divide(10000)
    return image.addBands(optical, None, True).updateMask(mask)


def add_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = image.normalizedDifference(["B8", "B11"]).rename("NDMI")
    evi = image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": image.select("B8"), "RED": image.select("B4"), "BLUE": image.select("B2")},
    ).rename("EVI")
    return image.addBands([ndvi, ndmi, evi])


def build_annual_composite(year: int, aoi: "ee.Geometry"):
    start = f"{year}-{DRY_SEASON[0]}"
    end = f"{year}-{DRY_SEASON[1]}"
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_COVER_MAX))
        .map(mask_s2_clouds)
    )
    n_scenes = collection.size().getInfo()
    print(f"  {year}: {n_scenes} cloud-free scenes found")

    composite = collection.select(S2_BANDS).median()
    composite = add_indices(composite)
    return composite.clip(aoi)

## 4. Run the export for each year (2020-2024) to Google Drive

In [ ]:
tasks = {}

for year in range(YEAR_START, YEAR_END + 1):
    print(f"Building composite for {year}...")
    composite = build_annual_composite(year, aoi)
    bands = S2_BANDS + ["NDVI", "EVI", "NDMI"]

    task = ee.batch.Export.image.toDrive(
        image=composite.select(bands).toFloat(),  # force Float32 on every band, avoids mixed-dtype export errors
        description=f"sentinel2_banten_{year}",
        folder=DRIVE_FOLDER,
        fileNamePrefix=f"sentinel2_banten_{year}",
        region=aoi,
        scale=EXPORT_SCALE,
        crs=EXPORT_CRS,
        maxPixels=1e13,
    )
    task.start()
    tasks[year] = task
    print(f"  Export task started: sentinel2_banten_{year} -> Google Drive folder '{DRIVE_FOLDER}'")

print("\nAll tasks submitted (this runs async, nothing to wait for here).")

## 5. Check export status

In [ ]:
# Run this cell whenever you want to check progress (safe to re-run until everything is COMPLETED)
for year, task in tasks.items():
    status = task.status()
    print(f"{year}: {status['state']}" + (f" - {status.get('error_message')}" if status.get("error_message") else ""))

## Once the export finishes

1. Open Google Drive, look for the **`GeoSuitability_Sentinel2`** folder
2. Download the `sentinel2_banten_<year>.tif` files (2020-2024 - note GEE may split a single year into several tiles if the file is large)
3. Move them into `datasets/raw/sentinel2gee/` in this project

Exports can take anywhere from a few minutes to a while depending on how busy GEE's servers are - check https://code.earthengine.google.com/tasks or just re-run the status cell above.